# ModelMaker3D Cloud (Hunyuan3D-2 on Colab Pro)

TripoSR が苦手な複雑な対象(建築など)を、Colab の GPU で **Hunyuan3D-2** を使って高品質に3D化する。
生成ごとの従量課金API ではなく、あなたの Colab の GPU を使う。

## 使い方
1. ランタイム → ランタイムのタイプを変更 → **L4 GPU + ハイメモリON**(推奨)を選び保存
   - L4/A100(Colab Pro)なら本物UVテクスチャ生成(paint)が動く。T4は形状のみに自動フォールバック。
   - 最重量級(建築/メカ)のときだけ A100 に切替。
2. 上から順にセルを実行する(全セル実行でOK)
3. 最後のセルが表示する **公開URL** (`https://xxxx.trycloudflare.com`) をコピー
4. Mac 側で一度だけ: `echo 'https://xxxx.trycloudflare.com' > ~/.modelmaker3d_cloud`
5. Mac で: `bash engine/make3d.sh cloud path/to/image.png` (このノートブックは実行したまま開いておく)

In [ ]:
# 0) GPU 確認
import torch
assert torch.cuda.is_available(), 'GPUランタイムを選択してください(ランタイム→タイプを変更→GPU)'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 1) Hunyuan3D-2 を clone してインストール(数分)
import os
if not os.path.exists('/content/Hunyuan3D-2'):
    !git clone https://github.com/Tencent/Hunyuan3D-2.git /content/Hunyuan3D-2
%cd /content/Hunyuan3D-2
!pip -q install -r requirements.txt
!pip -q install -e .
!pip -q install fastapi 'uvicorn[standard]' python-multipart rembg trimesh
# numpy/scipy をHunyuanと相性の良い版に固定(壊れ/AttributeError対策)。
# インストール後に版がズレると scipy 読み込みでクラッシュするため最後に強制統一する。
!pip -q install --force-reinstall --no-cache-dir "numpy==1.26.4" "scipy==1.13.1"
print('shapegen install done')
print('※ numpy を入れ替えたので、この後 ランタイム→セッションを再起動 してから全セル実行してください')

In [ ]:
# 2) テクスチャ生成(paint)コンポーネントのビルド(任意・失敗しても可)
#    失敗/省略した場合は『形状のみ』を返し、Mac側で入力画像を投影して色付けする。
%cd /content/Hunyuan3D-2
import subprocess
def try_build(path):
    try:
        print('building', path)
        subprocess.run('pip install -e .', shell=True, cwd=path, check=True)
        return True
    except Exception as e:
        print('build failed:', path, e); return False
try_build('/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer')
try_build('/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer')
print('paint build attempted')

In [ ]:
# 3) cloudflared を取得(公開URL用)
import os
if not os.path.exists('/content/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared
print('cloudflared ready')

In [ ]:
# 4) モデルを読み込み、FastAPI サーバを起動 → cloudflared で公開URLを表示
#    このセルは実行したままにしておくこと(サーバが動き続ける)。
import io, os, re, time, tempfile, threading, subprocess
import torch, trimesh
from PIL import Image
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import Response
import uvicorn

from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
from hy3dgen.rembg import BackgroundRemover

# メッシュ後処理(あれば使う): 浮遊/縮退面の除去でメッシュ品質を底上げ
try:
    from hy3dgen.shapegen import FloaterRemover, DegenerateFaceRemover
    _floater, _degen = FloaterRemover(), DegenerateFaceRemover()
except Exception as e:
    print('mesh cleaners unavailable:', e); _floater = _degen = None

# 標準モデル。既定サブフォルダ(hunyuan3d-dit-v2-0)で確実に読める。
# 軽量版: from_pretrained('tencent/Hunyuan3D-2mini', subfolder='hunyuan3d-dit-v2-mini')
MODEL_ID = 'tencent/Hunyuan3D-2'
print('loading shape pipeline...', MODEL_ID)
shape_pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(MODEL_ID)
try:
    shape_pipe.enable_flashvdm()  # あれば省メモリ/高速化
except Exception:
    pass
rembg = BackgroundRemover()

# paint(本物UVテクスチャ)を有効化。L4/A100(Colab Pro)推奨。
# T4等で載らない/OOMの場合は自動で『形状のみ』にフォールバック(クラッシュしない)。
ENABLE_PAINT = True
# paint テクスチャ解像度。L4=2048推奨、A100なら4096も可。
PAINT_TEX_RES = 2048
paint_pipe = None
if ENABLE_PAINT:
    # 新しい diffusers は Hunyuan の custom pipeline 読込に trust_remote_code=True が必須。
    # DiffusionPipeline.from_pretrained に既定で渡すようパッチ(paint 内部で使われる)。
    try:
        from diffusers import DiffusionPipeline as _DP
        if not getattr(_DP.from_pretrained, '_trc', False):
            _o = _DP.from_pretrained
            def _p(*a, **k):
                k.setdefault('trust_remote_code', True)
                return _o(*a, **k)
            _p._trc = True
            _DP.from_pretrained = _p
    except Exception as _e:
        print('trust_remote_code patch skip:', _e)
    try:
        from hy3dgen.texgen import Hunyuan3DPaintPipeline
        paint_pipe = Hunyuan3DPaintPipeline.from_pretrained(MODEL_ID)
        cfg = getattr(paint_pipe, 'config', None)
        if cfg is not None:
            for attr in ('render_size', 'texture_size', 'tex_resolution'):
                if hasattr(cfg, attr):
                    try: setattr(cfg, attr, PAINT_TEX_RES)
                    except Exception: pass
        print('paint pipeline ready (tex_res target=%d)' % PAINT_TEX_RES)
    except Exception as e:
        print('paint unavailable (形状のみで返す):', e)

app = FastAPI()

@app.get('/health')
def health():
    return {'ok': True, 'paint': paint_pipe is not None}

def _run_shape(img, steps, octree):
    # octree_resolution 対応版 → 非対応版 の順で安全に呼ぶ
    for kw in (dict(num_inference_steps=steps, octree_resolution=octree),
               dict(num_inference_steps=steps)):
        try:
            return shape_pipe(image=img, **kw)[0]
        except TypeError:
            continue
    return shape_pipe(image=img)[0]

@app.post('/gen')
def gen(image: UploadFile = File(...), texture: str = Form('auto'),
        steps: str = Form('50'), octree: str = Form('384')):
    raw = image.file.read()
    img = Image.open(io.BytesIO(raw)).convert('RGBA')
    img = rembg(img)
    try:
        octv = int(octree)
    except Exception:
        octv = 384
    mesh = _run_shape(img, int(steps), octv)
    # メッシュ後処理(あれば)
    for fn in (_floater, _degen):
        if fn is not None:
            try: mesh = fn(mesh)
            except Exception as e: print('cleaner skip:', e)
    textured = False
    if texture != 'shape' and paint_pipe is not None:
        try:
            mesh = paint_pipe(mesh, image=img)
            textured = True
        except Exception as e:
            print('paint failed, returning shape-only:', e)
            torch.cuda.empty_cache()
    path = tempfile.mktemp(suffix='.glb')
    mesh.export(path)
    data = open(path, 'rb').read()
    torch.cuda.empty_cache()
    return Response(content=data, media_type='model/gltf-binary', headers={'X-Textured': '1' if textured else '0'})

def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')
threading.Thread(target=run_server, daemon=True).start()
time.sleep(4)

proc = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in proc.stdout:
    print(line.rstrip())
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
print('\n' + '='*60)
print('  公開URL(これをMacの ~/.modelmaker3d_cloud に保存):')
print('   ', url)
print('  Mac:  echo "%s" > ~/.modelmaker3d_cloud' % url)
print('  Mac:  bash engine/make3d.sh cloud path/to/image.png')
print('='*60)
for line in proc.stdout:
    pass